# HTQ2 GTFS Pipeline — Secrets Setup
This notebook creates the `trafiklab` secrets scope and adds the required API keys for the Trafiklab.se GTFS Regional API.

**Prerequisites:** Get your API keys from [Trafiklab.se](https://www.trafiklab.se/) → My Applications → GTFS Regional.

Two keys are needed:
- `realtime-api-key` — for GTFS Realtime feeds (ServiceAlerts, TripUpdates, VehiclePositions)
- `static-api-key` — for GTFS Static downloads (schedule ZIP + extra DVJId mapping ZIP)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# Create the trafiklab secrets scope
try:
    w.secrets.create_scope(scope="trafiklab")
    print("✓ Secrets scope 'trafiklab' created successfully")
except Exception as e:
    if "already exists" in str(e).lower() or "ALREADY_EXISTS" in str(e):
        print("✓ Secrets scope 'trafiklab' already exists — skipping creation")
    else:
        raise e

## Add Your API Keys
Replace `YOUR_REALTIME_KEY` and `YOUR_STATIC_KEY` below with your actual Trafiklab API keys, then run the cell.

In [0]:
# ⚠️ Replace these with your actual Trafiklab API keys before running
REALTIME_API_KEY = "<YOUR KEY>"
STATIC_API_KEY = "<YOUR KEY>"

# Store in Databricks Secrets
w.secrets.put_secret(scope="trafiklab", key="realtime-api-key", string_value=REALTIME_API_KEY)
print("✓ Secret 'realtime-api-key' stored")

w.secrets.put_secret(scope="trafiklab", key="static-api-key", string_value=STATIC_API_KEY)
print("✓ Secret 'static-api-key' stored")

In [0]:
# Verify secrets are stored (values are never shown — only key names)
secrets = list(w.secrets.list_secrets(scope="trafiklab"))
print(f"Secrets in 'trafiklab' scope ({len(secrets)}):")
for s in secrets:
    print(f"  ✓ {s.key} (last updated: {s.last_updated_timestamp})")

if len(secrets) >= 2:
    print("\n✅ All secrets configured! The HTQ2 pipeline job can now authenticate with Trafiklab.se")
else:
    print("\n⚠️ Missing secrets — ensure both realtime-api-key and static-api-key are set")

## ✅ Done
The `trafiklab` secrets scope is ready. The pipeline job references these secrets via:
```python
dbutils.secrets.get(scope="trafiklab", key="realtime-api-key")
dbutils.secrets.get(scope="trafiklab", key="static-api-key")
```

After running the cells, navigate back to job [htq2_gtfs_pipeline_dev](#job/977100775249708) to continue setup.